[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/generic/generic.ipynb)

# Damped Oscillator in the GENERIC Formulation

This example computes how a mass on a spring slowed by a viscous damper converts its mechanical energy into heat in a thermal reservoir. Kinetic and elastic energy fall together while the reservoir's internal energy rises by the same amount, keeping the total energy constant as the reservoir's entropy grows until the mass comes to rest. The GENERIC formulation (General Equation for Non-Equilibrium Reversible-Irreversible Coupling) splits such an evolution into a reversible flow driven by total energy and an irreversible flow driven by entropy.

The algebra `q0 p0 s0` gives one degenerate direction each to the position `q`, the momentum `p` and the reservoir entropy `s`. A state is modelled as an antivector `position * q.dual() + momentum * p.dual() + entropy * s.dual()`, and a potential is a vector that pairs with changes of state. The energy potential combines the spring force along `q`, the velocity along `p` and the reservoir temperature along `s`, while the entropic potential is `s`. Two maps from potentials to rates of state change generate the motion. The skew Poisson map `q.dual() * (Potential & p.dual()) - p.dual() * (Potential & q.dual())` sends the energy potential to reversible oscillation and maps `s` to zero. The friction mobility is a positive dyad along the step that trades momentum for reservoir heat, sending `s` to viscous damping while mapping the energy potential to zero. Those two mutual null conditions keep total energy constant and make entropy production nonnegative.

In the matrix notation of [Öttinger, 2018](https://arxiv.org/abs/1810.08470) and the GENERIC literature on state $\mathbf x = (q, p, S)^{\mathsf T}$, the evolution $\dot{\mathbf x} = L\,\nabla E + M\,\nabla S$ with $\nabla E = (kq,\ p/m,\ T)^{\mathsf T}$ and $\nabla S = (0, 0, 1)^{\mathsf T}$ assembles the $3\times 3$ operators $L = \bigl[\begin{smallmatrix} 0 & 1 & 0 \\ -1 & 0 & 0 \\ 0 & 0 & 0 \end{smallmatrix}\bigr]$ and $M = \gamma T \bigl[\begin{smallmatrix} 0 & 0 & 0 \\ 0 & 1 & -p/(mT) \\ 0 & -p/(mT) & p^2/(m^2 T^2) \end{smallmatrix}\bigr]$ to satisfy $L\,\nabla S = \mathbf 0$ and $M\,\nabla E = \mathbf 0$, reducing to $m\ddot q = -kq - \gamma\dot q$ and $T\dot S = \gamma\dot q^2 \ge 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.thermodynamics.generic import render
from examples.util.animation import show_states

ga = Algebra("q0p0s0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                               # Scalar+
State = ga.gatype.antivector()                            # Antivector-
Potential = ga.gatype.vector()                            # Vector+
Map = ga.gatype((State, Potential))                       # State- <- Potential+

# Canonical Poisson map sending momentum slopes to position rates and position slopes to minus momentum rates.
poisson_map = (
    mv.q.dual() * (Potential & mv.p.dual()) - mv.p.dual() * (Potential & mv.q.dual())
)                                                         # [] State- <- Potential+
# Entropic potential pairing with a change of state to read the change in reservoir entropy.
entropy_potential = mv.s                                  # [] Potential+

## Energy potential and generator maps

The total energy sums kinetic energy, spring elastic energy and reservoir internal energy, where the reservoir has constant heat capacity so its temperature grows exponentially with its entropy. Applying `poisson_map` to `energy_potential` produces undamped harmonic motion, while applying `friction_mobility` to `entropy_potential` subtracts viscous drag from momentum and deposits the dissipated mechanical power as entropy in the reservoir. At constant total energy, reservoir entropy is determined by position and momentum alone through `surface_entropy`.

In [ ]:
mass, stiffness, friction = 1.0, 4.0, 0.35
thermal_capacity, reference_temperature = 2.0, 1.0
displacement = 2.0


def temperature(state: State) -> Scalar:
    """Absolute temperature of the thermal reservoir from its entropy and heat capacity."""
    return reference_temperature * ((mv.s & state) / thermal_capacity).exp()


def energies(state: State) -> tuple[Scalar, Scalar, Scalar]:
    """Kinetic, elastic and reservoir internal energy at `state`."""
    return (
        (mv.p & state).squared() / (2 * mass),
        stiffness * (mv.q & state).squared() / 2,
        thermal_capacity * temperature(state),
    )


def energy_potential(state: State) -> Potential:
    """Thermodynamic potential combining spring force along `q`, velocity along `p` and temperature along `s`."""
    return mv.q * (stiffness * (mv.q & state)) + mv.p * ((mv.p & state) / mass) + mv.s * temperature(state)


def friction_mobility(state: State) -> Map:
    """Positive dyad mapping potentials along the energy-conserving step from momentum into reservoir entropy."""
    reservoir_temperature = temperature(state)                                              # [...] Scalar+
    # Losing a unit of momentum at velocity v dissipates heat v into the reservoir, raising its entropy by v / T.
    dissipation_step = -mv.p.dual() + mv.s.dual() * ((mv.p & state) / (mass * reservoir_temperature))   # [...] State-
    return dissipation_step * (Potential & dissipation_step) * (friction * reservoir_temperature)


def rates(state: State) -> tuple[State, State]:
    """Reversible rate from the Poisson map on the energy potential and irreversible rate from friction mobility."""
    return poisson_map(energy_potential(state)), friction_mobility(state)(entropy_potential)


initial_state = mv.q.dual() * displacement + mv.p.dual() * 0.0 + mv.s.dual() * 0.0         # [] State-
initial_energy = sum(energies(initial_state))                                               # [] Scalar+


def surface_entropy(state: State) -> Scalar:
    """Reservoir entropy as a function of position and momentum when total energy equals `initial_energy`."""
    kinetic, elastic, _ = energies(state)
    internal = initial_energy - kinetic - elastic
    return thermal_capacity * (internal / (thermal_capacity * reference_temperature)).log()


render.draw(energies, surface_entropy, initial_state);

## Damped oscillation over time

Classical fourth-order Runge-Kutta integrates the sum of the reversible and irreversible rates forward in time, rendering one frame every two integration steps. This cell takes about ten seconds, nearly all of it drawing the animation.

In [ ]:
dt, n_steps_per_frame, n_frames, frame_duration_ms = 0.04, 2, 150, 40


def advance(state: State, step_dt: float) -> State:
    """State after one fourth-order Runge-Kutta step of the combined reversible and irreversible rates."""

    def rate(current: State) -> State:
        reversible, irreversible = rates(current)
        return reversible + irreversible

    k1 = rate(state)
    k2 = rate(state + k1 * (step_dt / 2))
    k3 = rate(state + k2 * (step_dt / 2))
    k4 = rate(state + k3 * step_dt)
    return state + (k1 + 2 * k2 + 2 * k3 + k4) * (step_dt / 6)


def release() -> Iterator[State]:
    """Yield the oscillator state at each frame."""
    state = initial_state
    for _ in range(n_frames):
        yield state
        for _ in range(n_steps_per_frame):
            state = advance(state, dt)


trajectory = show_states(
    release(), render.damping, energies, surface_entropy, n_steps_per_frame * dt,
    name="generic_thermodynamics", duration_ms=frame_duration_ms, n_frames=n_frames,
)

In the phase portrait on the left, teal ellipses mark contours of constant reservoir entropy at fixed total energy. Reversible motion carries the state along those ellipses while viscous damping pulls the trajectory inward across them, fastest when speed peaks at the origin. The middle chart shows entropy rising in steps that flatten whenever the mass reverses at its turning points. On the right chart, the state climbs the constant-energy curve as mechanical energy drains into the reservoir. Total energy remains `10`, starting with `8` in the spring and `2` in the reservoir, and ending near rest with all `10` in the reservoir at temperature `5`. The damper cylinder shifts from icy blue to red heat by blending the remaining warming fraction with its logarithm so the final approach to rest stays visible.

In [ ]:
states = stack(trajectory)                                                    # [n_frames] State-
final_energies = stack(energies(states[-1]))                                  # [3] Scalar+
np.set_printoptions(precision=4, suppress=True)
print("final kinetic, spring and internal energy:", final_energies.to_array())
print("final reservoir temperature:", temperature(states[-1]).to_array())

In [ ]:
# checks
kinetic, elastic, internal = energies(states)                                 # [n_frames] Scalar+, Scalar+, Scalar+
entropies = mv.s & states                                                     # [n_frames] Scalar+
np.testing.assert_allclose(((kinetic + elastic + internal) - initial_energy).kernel, 0, atol=1e-4)
assert np.all(entropies[1:] - entropies[:-1] >= -1e-8)
# The Poisson map annihilates the entropic potential, and the friction mobility annihilates the energy potential.
sample_state = mv.q.dual() * 0.7 - mv.p.dual() * 1.3 + mv.s.dual() * 0.4      # [] State-
np.testing.assert_allclose(poisson_map(entropy_potential).kernel, 0, atol=1e-6)
np.testing.assert_allclose(friction_mobility(sample_state)(energy_potential(sample_state)).kernel, 0, atol=1e-6)